In [14]:
%pip install soundfile
%pip install librosa

Note: you may need to restart the kernel to use updated packages.
  Using cached msgpack-1.1.2-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
Using cached msgpack-1.1.2-cp312-cp312-win_amd64.whl (72 kB)
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 26.5 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---------------- ----------------------- 15.7/38.1 MB 76.2 MB/s eta 0:00:01
   -------------------------------------- - 36.7/38.1 MB 89.7 MB/s eta 0:00:01
   ---------------------------------------- 38.1/38.1 MB 67.4 MB/s  0:00:00

   ---------- ----------------------------- 2/8 [llvmlite]
   ---------- ----------------------------- 2/8 [llvmlite]
   ---------- ----------------------------- 2/8 [llvmlite]
   ---------- ----------------------------- 2/8 [llvmlite]
   ---------- ----------------------------- 2/8 [llvmlite]
   ------------------------- -------------- 5/8 [pooch]
 

In [1]:
import torchaudio
torchaudio.set_audio_backend("soundfile")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-p

In [ ]:
import torch
import torchaudio
from tqdm import tqdm

# -------- CONFIG --------
SAMPLE_RATE = 16000

# MFCC (lightweight)
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=40,  # standard choice
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 64
    }
)

# Log-Mel Spectrogram (heavy model)
mel_spec_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=400,
    hop_length=160,
    n_mels=128
)

amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [5]:
def extract_features(waveform, mode="mfcc"):
    """
    waveform: Tensor [1, T]
    mode: 'mfcc' or 'mel'
    """
    
    if mode == "mfcc":
        features = mfcc_transform(waveform)  # [1, n_mfcc, time]
        
    elif mode == "mel":
        features = mel_spec_transform(waveform)  # [1, n_mels, time]
        features = amplitude_to_db(features)
        
    else:
        raise ValueError("Unknown mode")

    return features.squeeze(0)  # [feat_dim, time]

In [6]:
MAX_LEN = SAMPLE_RATE * 6  # 6 seconds


def fix_length(waveform):
    """
    waveform: [1, T]
    """
    T = waveform.shape[1]

    if T > MAX_LEN:
        # truncate
        waveform = waveform[:, :MAX_LEN]
    else:
        # pad
        pad_size = MAX_LEN - T
        waveform = torch.nn.functional.pad(waveform, (0, pad_size))

    return waveform

In [17]:
import librosa
import torch

def load_audio(path):
    # librosa loads reliably on Windows
    y, sr = librosa.load(path, sr=SAMPLE_RATE)

    # convert to torch
    waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)

    waveform = fix_length(waveform)

    return waveform

In [8]:
import pandas as pd
from torch.utils.data import Dataset
from pathlib import Path

# -------- PATHS --------
DATA_ROOT = Path("../Data/MELD.Audio")

# label mapping (important)
EMO_MAP = {
    "joy": "happiness"
}

# final label set (fixed order!)
LABELS = [
    "anger",
    "disgust",
    "fear",
    "happiness",
    "sadness",
    "surprise",
    "neutral"
]

label2idx = {l: i for i, l in enumerate(LABELS)}

In [32]:
class MELDAudioDataset(Dataset):
    def __init__(self, split="train", mode="mfcc"):
        self.split = split
        self.mode = mode

        # load CSV
        if split == "train":
            csv_path = "../Data/MELD.Raw/train/train_sent_emo.csv"
        elif split == "dev":
            csv_path = "../Data/MELD.Raw/dev_sent_emo.csv"
        else:
            csv_path = "../Data/MELD.Raw/test_sent_emo.csv"

        df = pd.read_csv(csv_path)

        # normalize labels
        df["Emotion"] = df["Emotion"].str.lower()
        df["Emotion"] = df["Emotion"].replace(EMO_MAP)

        self.audio_dir = DATA_ROOT / split

        # filter missing files
        valid_rows = []
        for _, row in df.iterrows():
            dia = row["Dialogue_ID"]
            utt = row["Utterance_ID"]

            filename = f"dia{dia}_utt{utt}.wav"
            path = self.audio_dir / filename

            if path.exists():
                valid_rows.append(row)

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        dia = row["Dialogue_ID"]
        utt = row["Utterance_ID"]

        filename = f"dia{dia}_utt{utt}.wav"
        path = self.audio_dir / filename

        waveform = load_audio(path)
        features = extract_features(waveform, self.mode)

        label = label2idx[row["Emotion"]]

        return features, label

In [34]:
train_ds = MELDAudioDataset("train")
print(len(train_ds))
x, y = train_ds[0]

9988


In [13]:
torchaudio.get_audio_backend()

C:\Users\Andrei\AppData\Local\Temp\ipykernel_1380\1746710290.py:1: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.get_audio_backend()


In [18]:
ds = MELDAudioDataset(split="train", mode="mfcc")

x, y = ds[0]

print(x.shape)  # [feat_dim, time]
print(y)        # int

torch.Size([40, 601])
6


Small CNN on MFCC (fast, low params, decent performance)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AudioCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(24000, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: [B, feat, time] → add channel
        x = x.unsqueeze(1)  # [B, 1, F, T]

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
model = AudioCNN()

x = torch.randn(1, 40, 601)
out = model.conv1(x.unsqueeze(1))
out = model.pool(out)
out = model.pool(model.conv2(out))
out = model.pool(model.conv3(out))

print(out.shape)

torch.Size([1, 64, 5, 75])


In [21]:
model = AudioCNN()

x = torch.randn(2, 40, 601)  # batch of 2
out = model(x)

print(out.shape)  # should be [2, 7]

torch.Size([2, 7])


## Data loader

In [22]:
from torch.utils.data import DataLoader

train_ds = MELDAudioDataset(split="train", mode="mfcc")
dev_ds   = MELDAudioDataset(split="dev", mode="mfcc")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=32, shuffle=False)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [25]:
model = AudioCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Training loop

In [27]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Evaluation

In [28]:
def evaluate(model, loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            preds = out.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

Train

In [41]:
EPOCHS = 10

for epoch in tqdm(range(EPOCHS)):
    train_loss = train_one_epoch(model, train_loader)
    val_acc = evaluate(model, dev_loader)

    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, val_acc={val_acc:.4f}")

TypeError: 'module' object is not callable. Did you mean: 'tqdm.tqdm(...)'?